# 😷 Face Mask Detector
A two-stage deep learning pipeline: OpenCV detects faces, MobileNetV2 classifies each face as **with mask** or **without mask**.

**New concept in this project:** Transfer Learning — using a model pretrained on 1.2M images instead of training from scratch.

**Run everything:** Runtime → Run all (`Ctrl+F9`)

| Step | What you will do |
|------|------------------|
| 1 | Install & import |
| 2 | Download dataset |
| 3 | Explore & visualize |
| 4 | Build data generators |
| 5 | Build model (transfer learning) |
| 6 | Phase 1 — train the head (base frozen) |
| 7 | Phase 2 — fine-tune top layers |
| 8 | Evaluate |
| 9 | Save the model |
| 10 | Test with your own photo (Gradio) |

## Step 1 — Install & Import Libraries

In [ ]:
!pip install tensorflow gradio opencv-python-headless seaborn pillow --quiet

import os
import random
import zipfile
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import cv2
import tensorflow as tf
from PIL import Image

from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import confusion_matrix, classification_report

IMG_SIZE   = 224    # MobileNetV2 expects 224x224
BATCH_SIZE = 32
SEED       = 42

print('✅ All libraries imported!')
print(f'   TensorFlow : {tf.__version__}')
print(f'   OpenCV     : {cv2.__version__}')

## Step 2 — Download the Dataset

In [ ]:
# Download the Face Mask Detection dataset from GitHub
# Contains: with_mask/ (690 images) and without_mask/ (686 images)
!wget -q https://github.com/chandrikadeb7/Face-Mask-Detection/archive/refs/heads/master.zip -O dataset.zip
!unzip -q dataset.zip

DATA_DIR = 'Face-Mask-Detection-master/dataset'

with_mask    = os.listdir(f'{DATA_DIR}/with_mask')
without_mask = os.listdir(f'{DATA_DIR}/without_mask')

print('📊 Dataset ready!')
print(f'   With mask    : {len(with_mask)} images')
print(f'   Without mask : {len(without_mask)} images')
print(f'   Total        : {len(with_mask) + len(without_mask)} images')

## Step 3 — Explore & Visualize

In [ ]:
def load_sample_images(folder, n=8):
    files = random.sample(os.listdir(folder), n)
    images = []
    for f in files:
        img = Image.open(os.path.join(folder, f)).convert('RGB').resize((IMG_SIZE, IMG_SIZE))
        images.append(np.array(img))
    return images

with_samples    = load_sample_images(f'{DATA_DIR}/with_mask')
without_samples = load_sample_images(f'{DATA_DIR}/without_mask')

fig, axes = plt.subplots(2, 8, figsize=(16, 5))
for i, img in enumerate(with_samples):
    axes[0, i].imshow(img)
    axes[0, i].axis('off')
    if i == 0: axes[0, i].set_title('With mask', color='green', fontsize=10, loc='left')
for i, img in enumerate(without_samples):
    axes[1, i].imshow(img)
    axes[1, i].axis('off')
    if i == 0: axes[1, i].set_title('Without mask', color='red', fontsize=10, loc='left')
plt.suptitle('Sample images from the dataset', y=1.02, fontsize=13)
plt.tight_layout()
plt.show()

# Class distribution
plt.figure(figsize=(5, 3))
plt.bar(['With mask', 'Without mask'], [len(with_mask), len(without_mask)],
        color=['#2ecc71', '#e74c3c'], edgecolor='white')
plt.title('Class distribution')
plt.tight_layout()
plt.show()
print('💡 Dataset is balanced — no class weighting needed.')

## Step 4 — Build Data Generators

In [ ]:
# ImageDataGenerator handles:
# 1. Loading images from folders
# 2. Resizing to 224x224
# 3. Applying MobileNetV2 preprocessing (scales pixels to -1..1)
# 4. Augmenting training images on the fly

train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    validation_split=0.2,
    rotation_range=20,
    zoom_range=0.15,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.15,
    horizontal_flip=True,
    fill_mode='nearest'
)

val_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    validation_split=0.2
)

train_gen = train_datagen.flow_from_directory(
    DATA_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='training',
    seed=SEED
)

val_gen = val_datagen.flow_from_directory(
    DATA_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='validation',
    seed=SEED
)

# Class mapping (0 = with_mask, 1 = without_mask)
CLASS_NAMES = {v: k for k, v in train_gen.class_indices.items()}
print(f'\nClass mapping: {CLASS_NAMES}')
print(f'Training   samples: {train_gen.samples}')
print(f'Validation samples: {val_gen.samples}')

## Step 5 — Build the Model (Transfer Learning)
> **What is transfer learning?** MobileNetV2 was already trained on 1.2 million images and learned to detect edges, textures, and shapes. We keep all that knowledge (freeze the base) and only train a small custom head on top of it for our specific task (mask vs no mask). This means we need far less data and training time.

In [ ]:
# Load MobileNetV2 WITHOUT its original top classifier
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

# Freeze the entire base — its weights won't change during Phase 1
base_model.trainable = False

# Build our custom classification head on top
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.5)(x)
output = Dense(1, activation='sigmoid')(x)  # 1 neuron: 0=with_mask, 1=without_mask

model = Model(inputs=base_model.input, outputs=output)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print(f'Total layers       : {len(model.layers)}')
print(f'Trainable layers   : {sum(1 for l in model.layers if l.trainable)}')
print(f'Frozen layers      : {sum(1 for l in model.layers if not l.trainable)}')

## Step 6 — Phase 1: Train the Head
> The base is frozen. Only our custom Dense layers learn. Fast training (~5 epochs).

In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss', patience=4,
    restore_best_weights=True, verbose=1
)

print('🔒 Phase 1: Training custom head (base frozen)...')
history1 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=10,
    callbacks=[early_stop],
    verbose=1
)

print(f'\n✅ Phase 1 complete — val accuracy: {max(history1.history["val_accuracy"]):.4f}')

## Step 7 — Phase 2: Fine-Tune
> Unfreeze the top 30 layers of MobileNetV2 and retrain with a very low learning rate. This gently adjusts the pretrained weights for our domain.

In [ ]:
# Unfreeze the top 30 layers of the base model
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

fine_tune_layers = sum(1 for l in model.layers if l.trainable)
print(f'🔓 Phase 2: Fine-tuning {fine_tune_layers} layers...')

# Use a much lower learning rate to avoid destroying pretrained weights
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss', factor=0.5,
    patience=2, min_lr=1e-7, verbose=1
)

history2 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=10,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

print(f'\n✅ Phase 2 complete — val accuracy: {max(history2.history["val_accuracy"]):.4f}')

## Step 8 — Evaluate

In [ ]:
# Combine history from both phases
acc  = history1.history['accuracy']     + history2.history['accuracy']
val  = history1.history['val_accuracy'] + history2.history['val_accuracy']
loss = history1.history['loss']         + history2.history['loss']
vloss= history1.history['val_loss']     + history2.history['val_loss']
split_ep = len(history1.history['loss'])

# Training curves
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, train_h, val_h, title in zip(
    axes,
    [acc, loss], [val, vloss],
    ['Accuracy', 'Loss']
):
    ax.plot(train_h, label='Train')
    ax.plot(val_h,   label='Validation')
    ax.axvline(split_ep - 1, color='gray', linestyle='--', linewidth=1, label='Fine-tune start')
    ax.set_title(title)
    ax.set_xlabel('Epoch')
    ax.legend()
plt.tight_layout()
plt.show()

# Final evaluation on validation set
loss_val, acc_val = model.evaluate(val_gen, verbose=0)
print(f'Final Accuracy : {acc_val*100:.2f}%')
print(f'Final Loss     : {loss_val:.4f}\n')

# Confusion matrix
val_gen.reset()
y_true = val_gen.classes
y_pred = (model.predict(val_gen, verbose=0) > 0.5).astype(int).flatten()

class_labels = [CLASS_NAMES[0], CLASS_NAMES[1]]
print(classification_report(y_true, y_pred, target_names=class_labels))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_labels, yticklabels=class_labels)
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

## Step 9 — Save the Model

In [ ]:
model.save('face_mask_detector.keras')
print('✅ Model saved as face_mask_detector.keras')

# To reload:
# model = load_model('face_mask_detector.keras')

## Step 10 — Test with Your Own Photo (Gradio)
> Upload any photo. OpenCV detects all faces in it, the model classifies each one, and the result is drawn with colored bounding boxes.

In [ ]:
import gradio as gr

# Download OpenCV Haar Cascade for face detection
!wget -q https://raw.githubusercontent.com/opencv/opencv/master/data/haarcascades/haarcascade_frontalface_default.xml
face_cascade = cv2.CascadeClassifier('haarcascade_frontalface_default.xml')

def detect_and_classify(image):
    if image is None:
        return None, 'Upload a photo first!'

    img_rgb = np.array(image)
    img_bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)
    gray    = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

    # Detect faces
    faces = face_cascade.detectMultiScale(
        gray, scaleFactor=1.1, minNeighbors=5, minSize=(60, 60)
    )

    output_img = img_rgb.copy()
    results    = []

    if len(faces) == 0:
        return image, 'No faces detected. Try a clearer photo with a visible face.'

    for (x, y, w, h) in faces:
        # Add padding around face crop
        pad   = int(0.1 * min(w, h))
        x1    = max(0, x - pad)
        y1    = max(0, y - pad)
        x2    = min(img_rgb.shape[1], x + w + pad)
        y2    = min(img_rgb.shape[0], y + h + pad)

        face_crop = img_rgb[y1:y2, x1:x2]
        face_pil  = Image.fromarray(face_crop).resize((IMG_SIZE, IMG_SIZE))
        face_arr  = preprocess_input(np.array(face_pil, dtype='float32'))
        face_arr  = np.expand_dims(face_arr, axis=0)

        # Classify
        prob      = float(model.predict(face_arr, verbose=0)[0][0])
        no_mask   = prob > 0.5
        label     = f'No mask ({prob*100:.0f}%)' if no_mask else f'Mask ({(1-prob)*100:.0f}%)'
        color     = (231, 76, 60) if no_mask else (46, 204, 113)  # Red or green

        # Draw bounding box and label
        cv2.rectangle(output_img, (x, y), (x+w, y+h), color, 3)
        text_y = y - 12 if y > 30 else y + h + 25
        cv2.rectangle(output_img, (x, text_y - 22), (x + w, text_y + 4), color, -1)
        cv2.putText(output_img, label, (x + 4, text_y),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255, 255, 255), 2)

        results.append(label)

    summary = f'{len(faces)} face(s) found:\n' + '\n'.join(f'  • {r}' for r in results)
    return Image.fromarray(output_img), summary


gr.Interface(
    fn=detect_and_classify,
    inputs=gr.Image(type='pil', label='Upload a photo'),
    outputs=[
        gr.Image(type='pil', label='Result'),
        gr.Textbox(label='Summary', lines=4)
    ],
    title='😷 Face Mask Detector',
    description='Upload a photo with one or more faces. Green box = mask on. Red box = no mask.',
    examples=[]
).launch(debug=False)